In [27]:
import pandas as pd
import numpy as np 
from scipy import stats as stat
from scipy.stats import bootstrap as bootstrap
import matplotlib.pyplot as plt
import seaborn as sns
from seaborn import catplot
import random as rd
import os
from sklearn.preprocessing import StandardScaler as ss
from sklearn.decomposition import PCA as pca
import math
from scipy.stats import linregress
import mne
%matplotlib qt
import os
from itertools import compress

import mne
from mne import datasets
import numpy as np
from nilearn.plotting import plot_design_matrix

import mne_nirs
import mne_bids
# Import StatsModels
import statsmodels.formula.api as smf
import matplotlib as mpl

# Import MNE processing
from mne.preprocessing.nirs import beer_lambert_law, optical_density

# Import MNE-BIDS processing
from mne_bids import BIDSPath, get_entity_vals, read_raw_bids,stats

from mne_nirs.channels import get_long_channels, get_short_channels, picks_pair_to_idx
from mne_nirs.datasets import fnirs_motor_group
from mne_nirs.experimental_design import make_first_level_design_matrix
from mne_nirs.io.fold import fold_channel_specificity

# Import MNE-NIRS processing
from mne_nirs.statistics import run_glm, statsmodels_to_results
from mne_nirs.visualisation import plot_glm_group_topo, plot_glm_surface_projection 
import os.path as op

from mne.datasets.testing import data_path
from mne.viz import set_3d_view

from sklearn.linear_model import LinearRegression
from statsmodels.tsa.ar_model import AutoReg
import statsmodels.api as sm
# Import common libraries
from collections import defaultdict
from copy import deepcopy
from itertools import compress
from pprint import pprint

# Import Plotting Library
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Import StatsModels
import statsmodels.formula.api as smf
from mne import Epochs, events_from_annotations, set_log_level
from mne.preprocessing.nirs import (
    beer_lambert_law,
    optical_density,
    scalp_coupling_index,
    temporal_derivative_distribution_repair,
)

# Import MNE processing
from mne.viz import plot_compare_evokeds

# Import MNE-BIDS processing
from mne_bids import BIDSPath, read_raw_bids

# Import MNE-NIRS processing
from mne_nirs.channels import get_long_channels, picks_pair_to_idx
from mne_nirs.datasets import fnirs_motor_group
from mne_nirs.signal_enhancement import enhance_negative_correlation


In [29]:
file = r"C:\Users\pkell\OneDrive\Documents\MATLAB\glm_result_group_v3.csv"
df1 = pd.read_csv(file)

print(df1.head())

   source  detector type cond      beta        se     tstat  dfe         p  \
0       1         1  hbo   x1 -0.352582  0.415807 -0.847946   60  0.399838   
1       1         1  hbo   x2  0.313775  0.417388  0.751758   60  0.455135   
2       1         1  hbr   x1  0.054004  0.143621  0.376018   60  0.708230   
3       1         1  hbr   x2  0.324774  0.143793  2.258620   60  0.027555   
4       1         2  hbo   x1 -0.404069  0.364774 -1.107724   60  0.272402   

          q  minDiscoverableChange  RelativePower  
0  0.613085               1.047589       0.243499  
1  0.659409               1.051573       0.242577  
2  0.839795               0.361841       0.704970  
3  0.115230               0.362275       0.704127  
4  0.471733               0.919017       0.277565  


In [31]:
# Define channels in each region of interest
left_frontal = [[1, 1], [1, 2], [2, 1], [2, 2], [2, 3], [3, 1], [3, 3], [4, 3], [4, 2], [4, 4], [5, 3], [5, 4]]
left_par = [[11, 9], [11, 8], [12, 8], [12, 10], [13, 8], [13, 9], [13, 10], [13, 11]]
right_frontal = [[6, 5], [6, 6], [7, 5], [7, 6], [7,7], [8, 5], [8,7], [9, 7], [9,6], [10,7]]
right_par = [[14, 12], [14, 13], [15, 12], [15, 14], [16, 12], [16, 13], [16, 14],[16,15]]

# Create filters for left and right pairs
left_frontal_filter = df1[["source", "detector"]].apply(tuple, axis=1).isin([tuple(pair) for pair in left_frontal])
left_par_filter = df1[["source", "detector"]].apply(tuple, axis=1).isin([tuple(pair) for pair in left_par])
right_frontal_filter = df1[["source", "detector"]].apply(tuple, axis=1).isin([tuple(pair) for pair in right_frontal])
right_par_filter = df1[["source", "detector"]].apply(tuple, axis=1).isin([tuple(pair) for pair in right_par])


# Separate DataFrames
df_lf = df1[left_frontal_filter]
df_rf = df1[right_frontal_filter]
df_lp = df1[left_par_filter]
df_rp = df1[right_par_filter]



In [33]:
# Separate DataFrames
df_lf = df_lf[df_lf['type'] == 'hbo']
df_lfx1 = df_lf[df_lf["cond"] == 'x1'] 
df_lfx2 = df_lf[df_lf["cond"] == 'x2'] 

df_rf = df_rf[df_rf['type'] == 'hbo']
df_rfx1 = df_rf[df_rf["cond"] == 'x1'] 
df_rfx2 = df_rf[df_rf["cond"] == 'x2'] 

df_lp = df_lp[df_lp['type'] == 'hbo']
df_lpx1 = df_lp[df_lp["cond"] == 'x1'] 
df_lpx2 = df_lp[df_lp["cond"] == 'x2'] 

df_rp = df_rp[df_rp['type'] == 'hbo']
df_rpx1 = df_rp[df_rp["cond"] == 'x1'] 
df_rpx2 = df_rp[df_rp["cond"] == 'x2'] 

In [10]:
## 1st ORDER Pairwise Correlation Matrix Code (Peasrson R on Raw Betas for each channel in ROI)
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx

# Combine the two DataFrames
combined_df = pd.concat([df_lpx1, df_lfx1, df_rfx1, df_rpx1], keys=['LPX1', 'LFX1', 'RFX1', 'RPX1'], names=['Region']) ## add right hemisphere
combined_df.reset_index(inplace=True)

# Drop rows where the column has specific values
values_to_remove = [20,36,40,44,88,89,80]
combined_df = combined_df[~combined_df['level_1'].isin(values_to_remove)]


# Step 1: Group beta values by Region
region_groups = combined_df.groupby('Region')['beta']

# Step 2: Create a DataFrame of beta values for each region
region_beta_df = pd.DataFrame({region: group.values for region, group in region_groups})

# Ensure all regions have the same number of beta values by padding with NaN if necessary
max_length = region_beta_df.apply(len).max()
region_beta_df = region_beta_df.apply(lambda col: col.reindex(range(max_length)), axis=0)

# Step 3: Compute the correlation matrix across regions
correlation_matrix = region_beta_df.corr()

# Step 4: Visualize the correlation matrix
if correlation_matrix.empty or correlation_matrix.isnull().all().all():
    print("The correlation matrix is empty or contains only NaN values. Please check your data.")
else:
    # Heatmap visualization
    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", cbar_kws={'label': 'Correlation Coefficient'})
    plt.title('Functional Connectivity Heatmap 1st Order')
    plt.xlabel('Regions')
    plt.ylabel('Regions')
    plt.show()

    # Network graph visualization
    graph = nx.from_pandas_adjacency(correlation_matrix)
    plt.figure(figsize=(10, 8))
    nx.draw(graph, with_labels=True, node_color='lightblue', node_size=500, font_size=10)
    plt.title('Functional Connectivity Network')
    plt.show()

In [16]:
combined_df

,Region,level_1,source,detector,type,cond,beta,se,tstat,dfe,p,q,minDiscoverableChange,RelativePower
0,LPX1,96,11,8,hbo,x1,0.962588,0.441305,2.181231,60,0.033096,0.129568,1.111830,0.229430
1,LPX1,100,11,9,hbo,x1,-0.327466,0.470137,-0.696533,60,0.488785,0.671167,1.184469,0.215360
2,LPX1,108,12,8,hbo,x1,1.203709,0.498659,2.413890,60,0.018852,0.093749,1.256330,0.203042
3,LPX1,112,12,10,hbo,x1,0.858246,0.463526,1.851558,60,0.069012,0.201559,1.167815,0.218431
4,LPX1,120,13,8,hbo,x1,0.781355,0.412123,1.895925,60,0.062789,0.186341,1.038310,0.245676
5,LPX1,124,13,9,hbo,x1,0.341584,0.338288,1.009741,60,0.316674,0.509131,0.852289,0.299297
6,LPX1,128,13,10,hbo,x1,1.169934,0.379122,3.085905,60,0.003069,0.031369,0.955166,0.267061
7,LPX1,132,13,11,hbo,x1,0.487185,0.293077,1.662310,60,0.101667,0.267238,0.738383,0.345468
8,LFX1,0,1,1,hbo,x1,-0.352582,0.415807,-0.847946,60,0.399838,0.613085,1.047589,0.243499
9,LFX1,4,1,2,hbo,x1,-0.404069,0.364774,-1.107724,60,0.272402,0.471733,0.919017,0.277565


In [9]:
## 2nd ORDER - Pairwise Correlation Matrix Code (Peasrson R on Raw Betas for each channel in ROI) 
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx

# Combine the two DataFrames
combined_df = pd.concat([df_lpx2, df_lfx2, df_rfx2, df_rpx2], keys=['LPX2', 'LFX2', 'RFX2', 'RPX2'], names=['Region']) ## add right hemisphere
combined_df.reset_index(inplace=True)

# Drop rows where the column has specific values
values_to_remove = [25, 37, 41, 45, 85, 89]
combined_df = combined_df[~combined_df['level_1'].isin(values_to_remove)]


# Step 1: Group beta values by Region
region_groups = combined_df.groupby('Region')['beta']

# Step 2: Create a DataFrame of beta values for each region
region_beta_df = pd.DataFrame({region: group.values for region, group in region_groups})

# Ensure all regions have the same number of beta values by padding with NaN if necessary
max_length = region_beta_df.apply(len).max()
region_beta_df = region_beta_df.apply(lambda col: col.reindex(range(max_length)), axis=0)

# Step 3: Compute the correlation matrix across regions
correlation_matrix = region_beta_df.corr()

# Step 4: Visualize the correlation matrix
if correlation_matrix.empty or correlation_matrix.isnull().all().all():
    print("The correlation matrix is empty or contains only NaN values. Please check your data.")
else:
    # Heatmap visualization
    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", cbar_kws={'label': 'Correlation Coefficient'})
    plt.title('Functional Connectivity Heatmap')
    plt.xlabel('Regions')
    plt.ylabel('Regions')
    plt.show()

    # Network graph visualization
    graph = nx.from_pandas_adjacency(correlation_matrix)
    plt.figure(figsize=(10, 8))
    nx.draw(graph, with_labels=True, node_color='lightblue', node_size=500, font_size=10)
    plt.title('Functional Connectivity Network')
    plt.show()


In [ ]:
combined_df

In [11]:
## 2nd ORDER Partial Correlation Analysis 
##- partial correlation computes the relationship between two regions' beta values after removing their shared variance with the covariate.

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx
from pingouin import partial_corr

# Combine the two DataFrames
combined_df = pd.concat([df_lpx2, df_lfx2, df_rfx2, df_rpx2], keys=['LPX2', 'LFX2', 'RFX2', 'RPX2'], names=['Region']) ## add right hemisphere
combined_df.reset_index(inplace=True)

# Drop rows where the column has specific values
values_to_remove = [25, 37, 41, 45, 85, 89]
combined_df = combined_df[~combined_df['level_1'].isin(values_to_remove)]


# Ensure the data has no NaN values after dropping rows
combined_df.dropna(subset=['beta'], inplace=True)

# Step 1: Group beta values by Region
region_groups = combined_df.groupby('Region')['beta']

# Step 2: Create a DataFrame of beta values for each region
region_beta_df = pd.DataFrame({region: group.values for region, group in region_groups})

# Ensure all regions have the same number of beta values by padding with NaN if necessary
max_length = region_beta_df.apply(len).max()
region_beta_df = region_beta_df.apply(lambda col: col.reindex(range(max_length)), axis=0)

# Add covariates (e.g., global mean as a covariate; you can change it to a relevant one)
region_beta_df['Global'] = region_beta_df.mean(axis=1)

# Step 3: Compute the partial correlation matrix
regions = region_beta_df.columns[:-1]  # Exclude the covariate column
partial_corr_matrix = pd.DataFrame(index=regions, columns=regions)

for region1 in regions:
    for region2 in regions:
        if region1 == region2:
            partial_corr_matrix.loc[region1, region2] = 1.0  # Diagonal is 1
        else:
            # Calculate partial correlation controlling for 'Global'
            partial_corr_result = partial_corr(
                data=region_beta_df,
                x=region1,
                y=region2,
                covar='Global'
            )
            partial_corr_matrix.loc[region1, region2] = partial_corr_result['r'].values[0]

# Convert to float
partial_corr_matrix = partial_corr_matrix.astype(float)

# Step 4: Visualize the partial correlation matrix
if partial_corr_matrix.empty or partial_corr_matrix.isnull().all().all():
    print("The partial correlation matrix is empty or contains only NaN values. Please check your data.")
else:
    # Heatmap visualization
    plt.figure(figsize=(10, 8))
    sns.heatmap(partial_corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", cbar_kws={'label': 'Partial Correlation Coefficient'})
    plt.title('Functional Connectivity Heatmap (Partial Correlation)')
    plt.xlabel('Regions')
    plt.ylabel('Regions')
    plt.show()

    # Network graph visualization
    graph = nx.from_pandas_adjacency(partial_corr_matrix)
    plt.figure(figsize=(10, 8))
    nx.draw(graph, with_labels=True, node_color='lightblue', node_size=500, font_size=10)
    plt.title('Functional Connectivity Network (Partial Correlation)')
    plt.show()


In [ ]:
region_beta_df

In [13]:
## 2nd ORDER SPEARMAN CORRELATION
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx

# Combine the two DataFrames
combined_df = pd.concat([df_lpx2, df_lfx2, df_rfx2, df_rpx2], keys=['LPX2', 'LFX2', 'RFX2', 'RPX2'], names=['Region']) ## add right hemisphere
combined_df.reset_index(inplace=True)

# Drop rows where the column has specific values
values_to_remove = [25, 37, 41, 45, 85, 89]
combined_df = combined_df[~combined_df['level_1'].isin(values_to_remove)]


# Step 1: Group beta values by Region
region_groups = combined_df.groupby('Region')['beta']

# Step 2: Create a DataFrame of beta values for each region
region_beta_df = pd.DataFrame({region: group.values for region, group in region_groups})

# Step 3: Compute the Spearman correlation matrix across regions
correlation_matrix = region_beta_df.corr(method='spearman')

# Step 4: Visualize the correlation matrix
if correlation_matrix.empty or correlation_matrix.isnull().all().all():
    print("The correlation matrix is empty or contains only NaN values. Please check your data.")
else:
    # Heatmap visualization
    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", cbar_kws={'label': 'Correlation Coefficient'})
    plt.title('Functional Connectivity Heatmap (Spearman Correlation)')
    plt.xlabel('Regions')
    plt.ylabel('Regions')
    plt.show()

    # Network graph visualization
    graph = nx.from_pandas_adjacency(correlation_matrix)
    plt.figure(figsize=(10, 8))
    nx.draw(graph, with_labels=True, node_color='lightblue', node_size=500, font_size=10)
    plt.title('Functional Connectivity Network')
    plt.show()


In [ ]:
region_beta_df

In [26]:
## 2nd ORDER CROSS CORRELATION 
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx

# Combine the two DataFrames
combined_df = pd.concat([df_lpx2, df_lfx2, df_rfx2, df_rpx2], keys=['LPX2', 'LFX2', 'RFX2', 'RPX2'], names=['Region']) 
combined_df.reset_index(inplace=True)

# Drop rows where the column has specific values
values_to_remove = [25, 37, 41, 45, 85, 89]
combined_df = combined_df[~combined_df['level_1'].isin(values_to_remove)]

# Step 1: Group beta values by Region
region_groups = combined_df.groupby('Region')['beta']

# Step 2: Create a DataFrame of beta values for each region
region_beta_df = pd.DataFrame({region: group.values for region, group in region_groups})

# Ensure all regions have the same number of beta values by padding with NaN if necessary
max_length = region_beta_df.apply(len).max()
region_beta_df = region_beta_df.apply(lambda col: col.reindex(range(max_length)), axis=0)

# Step 3: Compute cross-correlation matrix across regions
def compute_cross_correlation(series1, series2, lag=1):
    """Compute cross-correlation between two signals for a given lag."""
    return np.corrcoef(series1[:-lag], series2[lag:])[0, 1] if lag < len(series1) else np.nan

# Create an empty matrix for cross-correlation values
cross_correlation_matrix = pd.DataFrame(index=region_beta_df.columns, columns=region_beta_df.columns)

# Compute cross-correlation for each pair of regions
for region1 in region_beta_df.columns:
    for region2 in region_beta_df.columns:
        # Calculate the maximum cross-correlation at a specific lag (you can adjust lag as needed)
        cross_correlation_matrix.loc[region1, region2] = compute_cross_correlation(region_beta_df[region1], region_beta_df[region2], lag=1)

# Step 4: Visualize the cross-correlation matrix
if cross_correlation_matrix.empty or cross_correlation_matrix.isnull().all().all():
    print("The cross-correlation matrix is empty or contains only NaN values. Please check your data.")
else:
    # Heatmap visualization
    plt.figure(figsize=(10, 8))
    sns.heatmap(cross_correlation_matrix.astype(float), annot=True, cmap='coolwarm', fmt=".2f", cbar_kws={'label': 'Cross-correlation Coefficient'})
    plt.title('Functional Connectivity Heatmap (Cross-correlation)')
    plt.xlabel('Regions')
    plt.ylabel('Regions')
    plt.show()

    # Network graph visualization
    graph = nx.from_pandas_adjacency(cross_correlation_matrix.astype(float))
    plt.figure(figsize=(10, 8))
    nx.draw(graph, with_labels=True, node_color='lightblue', node_size=500, font_size=10)
    plt.title('Functional Connectivity Network (Cross-correlation)')
    plt.show()


In [28]:
## 2nd ORDER Granger Causality

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx
from statsmodels.tsa.stattools import grangercausalitytests

# Combine the two DataFrames
combined_df = pd.concat([df_lpx2, df_lfx2, df_rfx2, df_rpx2], keys=['LPX2', 'LFX2', 'RFX2', 'RPX2'], names=['Region'])
combined_df.reset_index(inplace=True)

# Drop rows where the column has specific values
values_to_remove = [25, 37, 41, 45, 85, 89]
combined_df = combined_df[~combined_df['level_1'].isin(values_to_remove)]


# Step 1: Group beta values by Region
region_groups = combined_df.groupby('Region')['beta']

# Step 2: Create a DataFrame of beta values for each region
region_beta_df = pd.DataFrame({region: group.values for region, group in region_groups})

# Ensure all regions have the same number of beta values by padding with NaN if necessary
max_length = region_beta_df.apply(len).max()
region_beta_df = region_beta_df.apply(lambda col: col.reindex(range(max_length)), axis=0)

# Step 3: Compute Granger Causality between regions
# Initialize a dictionary to hold the results
granger_matrix = pd.DataFrame(index=region_beta_df.columns, columns=region_beta_df.columns)

# For each pair of regions, perform Granger causality test
for region1 in region_beta_df.columns:
    for region2 in region_beta_df.columns:
        if region1 != region2:
            # Perform Granger causality test with a max lag of 5 (adjust as needed)
            gc_test_result = grangercausalitytests(region_beta_df[[region1, region2]], maxlag=5, verbose=False)
            # Extract the p-value from the results
            p_value = gc_test_result[5][0]['ssr_chi2test'][1]
            # If p-value < 0.05, store 1 (indicating Granger causality), else 0
            granger_matrix.loc[region1, region2] = 1 if p_value < 0.05 else 0

# Step 4: Visualize the Granger causality matrix
if granger_matrix.empty or granger_matrix.isnull().all().all():
    print("The Granger causality matrix is empty or contains only NaN values. Please check your data.")
else:
    # Heatmap visualization
    plt.figure(figsize=(10, 8))
    sns.heatmap(granger_matrix.astype(int), annot=True, cmap='Blues', cbar_kws={'label': 'Granger Causality'})
    plt.title('Granger Causality Heatmap')
    plt.xlabel('Regions')
    plt.ylabel('Regions')
    plt.show()

    # Network graph visualization
    graph = nx.from_pandas_adjacency(granger_matrix.astype(int))
    plt.figure(figsize=(10, 8))
    nx.draw(graph, with_labels=True, node_color='lightblue', node_size=500, font_size=10)
    plt.title('Granger Causality Network')
    plt.show()


C:\Users\pkell\anaconda3\Lib\site-packages\statsmodels\tsa\stattools.py:1545: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


ValueError: Insufficient observations. Maximum allowable lag is 1

ERROR: Could not find a version that satisfies the requirement pywt (from versions: none)
ERROR: No matching distribution found for pywt


In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pywt

# Combine the DataFrames (replace with actual ROI data frames)
combined_df = pd.concat([df_lpx2, df_lfx2, df_rfx2, df_rpx2], keys=['LPX2', 'LFX2', 'RFX2', 'RPX2'], names=['Region'])
combined_df.reset_index(inplace=True)

# # Drop rows with specific unwanted values
values_to_remove = [25, 37, 41, 45, 85, 89]
combined_df = combined_df[~combined_df['level_1'].isin(values_to_remove)]

# Step 1: Group beta values by ROI
region_groups = combined_df.groupby('Region')['beta']

# Step 2: Compute mean beta values for each region (ROI time series)
roi_beta_values = {region: group.values for region, group in region_groups}

# Ensure all ROIs have the same length by padding with NaN if necessary
max_length = max(len(values) for values in roi_beta_values.values())
roi_time_series = {region: np.pad(values, (0, max_length - len(values)), constant_values=np.nan)
                   for region, values in roi_beta_values.items()}

# Step 3: Define wavelet parameters
wavelet = 'cmor1.5-1.0'  # Complex Morlet wavelet
scales = np.arange(1, 128)  # Scale range for frequency decomposition
sampling_rate = 1.4  # Update this based on your experiment
sampling_period = 1 / sampling_rate

# Step 4: Compute wavelet coherence for each ROI pair
roi_keys = list(roi_time_series.keys())
coherence_results = {}

for i in range(len(roi_keys)):
    for j in range(i + 1, len(roi_keys)):
        region_1, region_2 = roi_keys[i], roi_keys[j]
        signal_1 = roi_time_series[region_1]
        signal_2 = roi_time_series[region_2]
        
        # Remove NaN values for coherence computation
        valid_idx = ~np.isnan(signal_1) & ~np.isnan(signal_2)
        signal_1 = signal_1[valid_idx]
        signal_2 = signal_2[valid_idx]
        
        # Compute continuous wavelet transforms
        coeffs_1, freqs = pywt.cwt(signal_1, scales, wavelet, sampling_period)
        coeffs_2, _ = pywt.cwt(signal_2, scales, wavelet, sampling_period)
        
        # Compute wavelet coherence (retain 2D time-frequency structure)
        num = np.abs(coeffs_1 * np.conj(coeffs_2))**2
        den = (np.abs(coeffs_1)**2) * (np.abs(coeffs_2)**2)
        coherence_matrix = num / den  # Retain 2D structure
        
        # Store the result
        coherence_results[(region_1, region_2)] = (coherence_matrix, freqs)

# Step 5: Plot wavelet coherence for each ROI pair
for (region_1, region_2), (coherence_matrix, freqs) in coherence_results.items():
    time = np.arange(coherence_matrix.shape[1]) / sampling_rate  # Adjust time array
    block_duration = 36  # Expected block length in seconds
    max_time_idx = int(block_duration * sampling_rate)  # Number of samples in 36 seconds
    
    plt.figure(figsize=(10, 6))
    plt.contourf(time[:max_time_idx], freqs, coherence_matrix[:, :max_time_idx], levels=100, cmap='viridis')
    plt.colorbar(label='Wavelet Coherence')
    plt.title(f'Wavelet Coherence: {region_1} vs {region_2}')
    plt.xlabel('Time (s)')
    plt.ylabel('Frequency (Hz)')
    plt.yscale('log')  # Log scale for frequency
    plt.show()


In [22]:
# Combine the two DataFrames
combined_df = pd.concat([df_lpx1, df_lfx1, df_rfx1, df_rpx1], keys=['LPX1', 'LFX1', 'RFX1', 'RPX1'], names=['Region']) ## add right hemisphere
combined_df.reset_index(inplace=True)

# Drop rows where the column has specific values
values_to_remove = [20,36,40,44,88,89,80]
combined_df = combined_df[~combined_df['level_1'].isin(values_to_remove)]

# Ensure the data has no NaN values after dropping rows
combined_df.dropna(subset=['beta'], inplace=True)

# Step 1: Group beta values by Region
region_groups = combined_df.groupby('Region')['beta']

# Step 2: Create a DataFrame of beta values for each region
region_beta_df = pd.DataFrame({region: group.values for region, group in region_groups})

# Ensure all regions have the same number of beta values by padding with NaN if necessary
max_length = region_beta_df.apply(len).max()
region_beta_df = region_beta_df.apply(lambda col: col.reindex(range(max_length)), axis=0)

# Add covariates (e.g., global mean as a covariate; you can change it to a relevant one)
region_beta_df['Global'] = region_beta_df.mean(axis=1)

# Step 3: Compute the partial correlation matrix
regions = region_beta_df.columns[:-1]  # Exclude the covariate column
partial_corr_matrix = pd.DataFrame(index=regions, columns=regions)

for region1 in regions:
    for region2 in regions:
        if region1 == region2:
            partial_corr_matrix.loc[region1, region2] = 1.0  # Diagonal is 1
        else:
            # Calculate partial correlation controlling for 'Global'
            partial_corr_result = partial_corr(
                data=region_beta_df,
                x=region1,
                y=region2,
                covar='Global'
            )
            partial_corr_matrix.loc[region1, region2] = partial_corr_result['r'].values[0]

# Convert to float
partial_corr_matrix = partial_corr_matrix.astype(float)

# Step 4: Visualize the partial correlation matrix
if partial_corr_matrix.empty or partial_corr_matrix.isnull().all().all():
    print("The partial correlation matrix is empty or contains only NaN values. Please check your data.")
else:
    # Heatmap visualization
    plt.figure(figsize=(10, 8))
    sns.heatmap(partial_corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", cbar_kws={'label': 'Partial Correlation Coefficient'})
    plt.title('Functional Connectivity Heatmap (Partial Correlation)')
    plt.xlabel('Regions')
    plt.ylabel('Regions')
    plt.show()

    # Network graph visualization
    graph = nx.from_pandas_adjacency(partial_corr_matrix)
    plt.figure(figsize=(10, 8))
    nx.draw(graph, with_labels=True, node_color='lightblue', node_size=500, font_size=10)
    plt.title('Functional Connectivity Network (Partial Correlation)')
    plt.show()



In [24]:
# Combine the two DataFrames
combined_df = pd.concat([df_lpx1, df_lfx1, df_rfx1, df_rpx1], keys=['LPX1', 'LFX1', 'RFX1', 'RPX1'], names=['Region']) ## add right hemisphere
combined_df.reset_index(inplace=True)

# Drop rows where the column has specific values
values_to_remove = [20,36,40,44,88,89,80]
combined_df = combined_df[~combined_df['level_1'].isin(values_to_remove)]

# Ensure the data has no NaN values after dropping rows
combined_df.dropna(subset=['beta'], inplace=True)


# Step 1: Group beta values by Region
region_groups = combined_df.groupby('Region')['beta']

# Step 2: Create a DataFrame of beta values for each region
region_beta_df = pd.DataFrame({region: group.values for region, group in region_groups})

# Step 3: Compute the Spearman correlation matrix across regions
correlation_matrix = region_beta_df.corr(method='spearman')

# Step 4: Visualize the correlation matrix
if correlation_matrix.empty or correlation_matrix.isnull().all().all():
    print("The correlation matrix is empty or contains only NaN values. Please check your data.")
else:
    # Heatmap visualization
    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", cbar_kws={'label': 'Correlation Coefficient'})
    plt.title('Functional Connectivity Heatmap (Spearman Correlation)')
    plt.xlabel('Regions')
    plt.ylabel('Regions')
    plt.show()

    # Network graph visualization
    graph = nx.from_pandas_adjacency(correlation_matrix)
    plt.figure(figsize=(10, 8))
    nx.draw(graph, with_labels=True, node_color='lightblue', node_size=500, font_size=10)
    plt.title('Functional Connectivity Network')
    plt.show()

The correlation matrix is empty or contains only NaN values. Please check your data.


C:\Users\pkell\AppData\Local\Temp\ipykernel_32952\3142205400.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_lfx2['Region'] = 'Left Frontal'
C:\Users\pkell\AppData\Local\Temp\ipykernel_32952\3142205400.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_lpx2['Region'] = 'Left Parietal'
C:\Users\pkell\AppData\Local\Temp\ipykernel_32952\3142205400.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value inste